# Staging data quality: what we enforce and what we warn about

Case study section 2.2 lists four defect classes that can live inside a single record. This notebook checks each one against our data. Each check is separated by who owns the failure.

**The ownership rule.** A staging row has two parts:

1. **The envelope: Decade's.** Every delivery metadata column Decade's ingestion produces (`snapshot_id`, `investment_id`, `snapshot_created_at`, `institution_id`, `institution_name`, `party_id`, `account_id`, `connection_id`, `ingested_at`), plus the grain. If any of this breaks, the bug is in Decade's pipeline. The dbt tests run at `severity: error` and the build halts.
2. **The payload: the provider's.** Everything flattened out of `payload_json`. The spec says what it should look like. An institution respecting the spec is a hope, not a guarantee. These checks run at `severity: warn` with `store_failures: true`. The build keeps going, the counts stay visible, and the failing rows land in `main_dbt_test__audit.*` as the provider defect ledger.

Both sets live in `models/canonical/staging/_staging_quality.yml`.

| Section | Owner | Check | dbt severity |
|---------|-------|-------|--------------|
| E | Decade | Envelope fields present, grain unique | error |
| D1 | Provider | Required field arriving empty | warn |
| D2 | Provider | Right concept, wrong form | warn |
| D3 | Provider | Legal value outside the enumeration | warn |
| D4 | Provider | Field contradicts another field in the same payload | warn |
| A | Provider | Attribution: which institution sends each defect | (analysis) |

This notebook proves and explains the defects. Fixing happens downstream. Staging detects, intermediate resolves, the canonical contract guarantees.

In [1]:
import os
import duckdb
import pandas as pd

# staging views read data/raw/*.parquet relative to the repo root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

con = duckdb.connect("warehouse.duckdb", read_only=True)

DETAIL_TABLES = [
    "stg_openfinance__bank_fixed_incomes_positions_detail",
    "stg_openfinance__credit_fixed_incomes_positions_detail",
    "stg_openfinance__funds_positions_detail",
    "stg_openfinance__treasure_titles_positions_detail",
    "stg_openfinance__variable_incomes_positions_detail",
]
for t in DETAIL_TABLES:
    n = con.execute(f"SELECT count(*) FROM {t}").fetchone()[0]
    print(f"{t.replace('stg_openfinance__', ''):45s} {n:>10,} rows")

bank_fixed_incomes_positions_detail               64,535 rows
credit_fixed_incomes_positions_detail              5,447 rows
funds_positions_detail                             8,355 rows
treasure_titles_positions_detail                   9,138 rows
variable_incomes_positions_detail                 16,479 rows


## E: the envelope, checks we own

The envelope is every column our ingestion attaches around the provider payload: `snapshot_id`, `investment_id`, `snapshot_created_at`, `institution_id`, `institution_name`, `party_id`, `account_id`, `connection_id`, `ingested_at`. If any of them is missing, the row cannot be deduplicated, attributed, or processed incrementally. If the grain `(snapshot_id, investment_id)` repeats, the incremental delete+insert double-writes.

These must all be zero. That is why they are safe to enforce as `severity: error`. A nonzero count means our ingestion broke, and the build should stop.

In [2]:
ENVELOPE = [
    "snapshot_id", "investment_id", "snapshot_created_at",
    "institution_id", "institution_name", "party_id",
    "account_id", "connection_id", "ingested_at",
]

rows = []
for table in DETAIL_TABLES:
    nulls = ", ".join(
        f"count(*) FILTER (WHERE {c} IS NULL) AS null_{c}" for c in ENVELOPE
    )
    res = con.execute(f"""
        SELECT
            {nulls},
            count(*) - count(DISTINCT snapshot_id || '|' || investment_id) AS duplicate_grain
        FROM {table}
    """).fetchdf()
    r = res.iloc[0].to_dict()
    r["product"] = table.replace("stg_openfinance__", "").replace("_positions_detail", "")
    rows.append(r)

env = pd.DataFrame(rows).set_index("product")
print(env.T.to_string())

violations = int(env.to_numpy().sum())
assert violations == 0, f"envelope integrity broken: {violations} violations, our ingestion has a bug"
print(f"\nenvelope clean across all tables: safe to enforce as severity error")

product                   bank_fixed_incomes  credit_fixed_incomes  funds  treasure_titles  variable_incomes
null_snapshot_id                           0                     0      0                0                 0
null_investment_id                         0                     0      0                0                 0
null_snapshot_created_at                   0                     0      0                0                 0
null_institution_id                        0                     0      0                0                 0
null_institution_name                      0                     0      0                0                 0
null_party_id                              0                     0      0                0                 0
null_account_id                            0                     0      0                0                 0
null_connection_id                         0                     0      0                0                 0
null_ingested_at   

## D1: required field arriving empty

The OFB spec marks certain fields as required. The staging models tag those same fields with `required=true` in the `payload_field` macro. Here we count how often a provider left them empty anyway.

Only fields with at least one empty value are shown.

In [3]:
REQUIRED = {
    "stg_openfinance__bank_fixed_incomes_positions_detail": [
        "issuer_cnpj", "investment_type", "indexer", "issue_unit_price",
        "issue_unit_price_currency", "due_date", "issue_date",
        "purchase_date", "grace_period_date",
    ],
    "stg_openfinance__credit_fixed_incomes_positions_detail": [
        "investment_type", "tax_exempt", "indexer", "issue_unit_price",
        "issue_unit_price_currency", "issue_date", "due_date",
        "voucher_payment_indicator", "purchase_date",
    ],
    "stg_openfinance__funds_positions_detail": ["fund_name", "fund_cnpj"],
    "stg_openfinance__treasure_titles_positions_detail": [
        "isin_code", "product_name", "indexer", "rate_periodicity",
        "calculation", "due_date", "purchase_date", "voucher_payment_indicator",
    ],
    "stg_openfinance__variable_incomes_positions_detail": ["isin_code", "ticker"],
}

rows = []
for table, cols in REQUIRED.items():
    selects = ", ".join(
        f"count(*) FILTER (WHERE {c} IS NULL) AS {c}" for c in cols
    )
    res = con.execute(f"SELECT count(*) AS total, {selects} FROM {table}").fetchdf()
    total = int(res["total"].iloc[0])
    for c in cols:
        empty = int(res[c].iloc[0])
        if empty > 0:
            rows.append({
                "product": table.replace("stg_openfinance__", "").replace("_positions_detail", ""),
                "field": c,
                "empty": empty,
                "total": total,
                "pct": round(100 * empty / total, 2),
            })

d1 = pd.DataFrame(rows)
print(d1.to_string(index=False))
assert len(d1) > 0, "D1: expected at least one required field arriving empty"

             product         field  empty  total  pct
  bank_fixed_incomes       indexer   1052  64535 1.63
  bank_fixed_incomes purchase_date   1322  64535 2.05
credit_fixed_incomes       indexer    144   5447 2.64


## D2: right concept, wrong form

The value means the right thing but is written the wrong way. The case study's own example is a tax identifier with a decimal tail. A CNPJ must be exactly 14 digits. An ISIN must be 2 letters, 9 alphanumerics, and a check digit.

Dates have a wrong form too. `0001-01-01` is .NET's `DateTime.MinValue`: a missing purchase date serialized as a real-looking date. It casts to DATE without complaint, so only knowing the placeholder catches it.

Empty values are excluded here. They already belong to D1.

In [4]:
CNPJ = "^[0-9]{14}$"
ISIN = "^[A-Z]{2}[A-Z0-9]{9}[0-9]$"
DATE_PLACEHOLDER = "DATE '0001-01-01'"  # .NET DateTime.MinValue: missing date in disguise

FORMATS = [
    ("stg_openfinance__bank_fixed_incomes_positions_detail",   "issuer_cnpj", CNPJ),
    ("stg_openfinance__credit_fixed_incomes_positions_detail", "debtor_cnpj", CNPJ),
    ("stg_openfinance__funds_positions_detail",                "fund_cnpj",   CNPJ),
    ("stg_openfinance__variable_incomes_positions_detail",     "issuer_cnpj", CNPJ),
    ("stg_openfinance__treasure_titles_positions_detail",      "isin_code",   ISIN),
    ("stg_openfinance__variable_incomes_positions_detail",     "isin_code",   ISIN),
]

PLACEHOLDER_DATES = [
    ("stg_openfinance__bank_fixed_incomes_positions_detail",   "purchase_date"),
    ("stg_openfinance__credit_fixed_incomes_positions_detail", "purchase_date"),
    ("stg_openfinance__treasure_titles_positions_detail",      "purchase_date"),
]

rows = []
for table, col, pattern in FORMATS:
    bad = con.execute(f"""
        SELECT {col} AS bad_value, count(*) AS n
        FROM {table}
        WHERE {col} IS NOT NULL AND NOT regexp_matches({col}, '{pattern}')
        GROUP BY 1 ORDER BY 2 DESC LIMIT 3
    """).fetchdf()
    for _, r in bad.iterrows():
        rows.append({
            "product": table.replace("stg_openfinance__", "").replace("_positions_detail", ""),
            "field": col,
            "bad_value": r["bad_value"],
            "n": int(r["n"]),
        })

for table, col in PLACEHOLDER_DATES:
    n = con.execute(
        f"SELECT count(*) FROM {table} WHERE {col} = {DATE_PLACEHOLDER}"
    ).fetchone()[0]
    if n > 0:
        rows.append({
            "product": table.replace("stg_openfinance__", "").replace("_positions_detail", ""),
            "field": col,
            "bad_value": "0001-01-01",
            "n": int(n),
        })

d2 = pd.DataFrame(rows)
print(d2.to_string(index=False))
assert len(d2) > 0, "D2: expected at least one value in the wrong form"

           product         field         bad_value    n
bank_fixed_incomes   issuer_cnpj 92894922000108.00 1370
   treasure_titles     isin_code                    404
  variable_incomes     isin_code                    906
bank_fixed_incomes purchase_date        0001-01-01 1538
   treasure_titles purchase_date        0001-01-01  413


## D3: legal value outside the enumeration

The OFB spec closes some fields to a fixed list of values. The `accepted_values` tests in `models/staging/_staging_quality.yml` enforce this. The cell below reads those tests directly, then scans every one against the warehouse. That way the notebook tracks enforced coverage instead of a hand-picked subset.

A provider writing `IPC-A` instead of `IPCA` sends a value that looks reasonable but is outside the list. That would break any downstream logic matching on the enum.

In [ ]:
import yaml

QUALITY_YML = "models/staging/_staging_quality.yml"

def enum_tests(doc):
    for m in doc.get("models", []):
        for t in m.get("data_tests", []) or []:
            if isinstance(t, dict) and "accepted_values" in t:
                av = t["accepted_values"]
                yield m["name"], av["column_name"], tuple(av["values"])
        for c in m.get("columns", []) or []:
            for t in c.get("data_tests", []) or []:
                if isinstance(t, dict) and "accepted_values" in t:
                    yield m["name"], c["name"], tuple(t["accepted_values"]["values"])

with open(QUALITY_YML) as fh:
    ENUMS = list(enum_tests(yaml.safe_load(fh)))

rows = []
for table, col, allowed in ENUMS:
    bad = con.execute(f"""
        SELECT {col} AS illegal_value, count(*) AS n
        FROM {table}
        WHERE {col} IS NOT NULL AND {col} NOT IN {allowed}
        GROUP BY 1 ORDER BY 2 DESC
    """).fetchdf()
    for _, r in bad.iterrows():
        rows.append({
            "model": table.replace("stg_openfinance__", ""),
            "field": col,
            "illegal_value": r["illegal_value"],
            "n": int(r["n"]),
        })

d3 = pd.DataFrame(rows)
print(f"enums enforced: {len(ENUMS)}")
if d3.empty:
    print("no illegal values in any enum today")
else:
    print(d3.to_string(index=False))
assert len(d3) > 0, "D3: expected at least one value outside its enumeration"

## D4: field contradicts another field in the same payload

Each field is valid on its own. The defect only appears when two fields are compared. A title cannot be purchased before it was issued, it cannot be issued after it matures, and it cannot be purchased after it matures.

The OFB schema cannot catch this class. No single-field rule fails.

A first pass found 1,538 bank rows "purchased before issued". Every one carried `purchaseDate = 0001-01-01`, the D2 placeholder above tripping a cross-field rule by accident. With placeholder rows excluded, every date-ordering rule passes. This dataset has no genuine intra-record contradiction. The checks stay because the class is real even when today's count is zero.

In [6]:
# purchase_date predicates exclude the D2 placeholder so this section only
# counts contradictions between two real dates
NOT_PLACEHOLDER = f"purchase_date <> {DATE_PLACEHOLDER}"

CONTRADICTIONS = [
    ("stg_openfinance__bank_fixed_incomes_positions_detail",   f"{NOT_PLACEHOLDER} AND purchase_date < issue_date", "purchased before issued"),
    ("stg_openfinance__bank_fixed_incomes_positions_detail",   "issue_date > due_date",                          "issued after maturity"),
    ("stg_openfinance__bank_fixed_incomes_positions_detail",   f"{NOT_PLACEHOLDER} AND purchase_date > due_date",   "purchased after maturity"),
    ("stg_openfinance__credit_fixed_incomes_positions_detail", f"{NOT_PLACEHOLDER} AND purchase_date < issue_date", "purchased before issued"),
    ("stg_openfinance__credit_fixed_incomes_positions_detail", "issue_date > due_date",                          "issued after maturity"),
    ("stg_openfinance__credit_fixed_incomes_positions_detail", f"{NOT_PLACEHOLDER} AND purchase_date > due_date",   "purchased after maturity"),
    ("stg_openfinance__treasure_titles_positions_detail",      f"{NOT_PLACEHOLDER} AND purchase_date > due_date",   "purchased after maturity"),
]

rows = []
for table, violation, label in CONTRADICTIONS:
    n = con.execute(f"SELECT count(*) FROM {table} WHERE {violation}").fetchone()[0]
    rows.append({
        "product": table.replace("stg_openfinance__", "").replace("_positions_detail", ""),
        "contradiction": label,
        "rule_violated": violation,
        "n": int(n),
    })

d4 = pd.DataFrame(rows)
print(d4.to_string(index=False))
assert d4["n"].sum() == 0, "D4: a genuine intra-record contradiction appeared, investigate"
print("\nno genuine contradictions: the 1,538 hits of the first pass were all the D2 placeholder")

             product            contradiction                                                     rule_violated  n
  bank_fixed_incomes  purchased before issued purchase_date <> DATE '0001-01-01' AND purchase_date < issue_date  0
  bank_fixed_incomes    issued after maturity                                             issue_date > due_date  0
  bank_fixed_incomes purchased after maturity   purchase_date <> DATE '0001-01-01' AND purchase_date > due_date  0
credit_fixed_incomes  purchased before issued purchase_date <> DATE '0001-01-01' AND purchase_date < issue_date  0
credit_fixed_incomes    issued after maturity                                             issue_date > due_date  0
credit_fixed_incomes purchased after maturity   purchase_date <> DATE '0001-01-01' AND purchase_date > due_date  0
     treasure_titles purchased after maturity   purchase_date <> DATE '0001-01-01' AND purchase_date > due_date  0

no genuine contradictions: the 1,538 hits of the first pass were all the D2 pla

## A: attribution, which institution sends each defect

A warn count alone says something is wrong. To act on it (open a ticket with the provider, add a mapping rule, decide quarantine) we need to know who sends it. Every staging row carries `institution_name` in the envelope, so attribution is one GROUP BY away.

This is the same question the stored failures in `main_dbt_test__audit.*` answer after every dbt build. Here we compute it live for the biggest defects.

In [7]:
DEFECTS = [
    ("stg_openfinance__bank_fixed_incomes_positions_detail",   "D1 empty indexer",           "indexer IS NULL"),
    ("stg_openfinance__bank_fixed_incomes_positions_detail",   "D1 empty purchase_date",     "purchase_date IS NULL"),
    ("stg_openfinance__bank_fixed_incomes_positions_detail",   "D2 cnpj wrong form",         "issuer_cnpj IS NOT NULL AND NOT regexp_matches(issuer_cnpj, '^[0-9]{14}$')"),
    ("stg_openfinance__variable_incomes_positions_detail",     "D2 isin wrong form",         "isin_code IS NOT NULL AND NOT regexp_matches(isin_code, '^[A-Z]{2}[A-Z0-9]{9}[0-9]$')"),
    ("stg_openfinance__bank_fixed_incomes_positions_detail",   "D2 placeholder purchase_date",  f"purchase_date = {DATE_PLACEHOLDER}"),
    ("stg_openfinance__treasure_titles_positions_detail",      "D2 placeholder purchase_date",  f"purchase_date = {DATE_PLACEHOLDER}"),
    ("stg_openfinance__bank_fixed_incomes_positions_detail",   "D3 indexer IPC-A",           "indexer = 'IPC-A'"),
]

rows = []
for table, label, condition in DEFECTS:
    res = con.execute(f"""
        SELECT institution_name, count(*) AS n
        FROM {table}
        WHERE {condition}
        GROUP BY 1 ORDER BY 2 DESC
    """).fetchdf()
    for _, r in res.iterrows():
        rows.append({"defect": label, "institution": r["institution_name"], "n": int(r["n"])})

attr = pd.DataFrame(rows)
print(attr.to_string(index=False))

per_defect = attr.groupby("defect")["institution"].nunique()
print("\ninstitutions involved per defect:")
print(per_defect.to_string())

                   defect     institution   n
         D1 empty indexer          Nubank 395
         D1 empty indexer            Itau 227
         D1 empty indexer   Banco XP S.A. 168
         D1 empty indexer     BTG Banking  80
         D1 empty indexer         C6 Bank  79
         D1 empty indexer  Banco Inter PF  52
         D1 empty indexer          PicPay  38
         D1 empty indexer Banco do Brasil  13
   D1 empty purchase_date          Nubank 416
   D1 empty purchase_date            Itau 365
   D1 empty purchase_date   Banco XP S.A. 240
   D1 empty purchase_date     BTG Banking 134
   D1 empty purchase_date         C6 Bank 123
   D1 empty purchase_date Banco do Brasil  16
   D1 empty purchase_date          PicPay  15
   D1 empty purchase_date  Banco Inter PF  13
       D2 cnpj wrong form          Nubank 493
       D2 cnpj wrong form            Itau 273
       D2 cnpj wrong form   Banco XP S.A. 203
       D2 cnpj wrong form     BTG Banking 168
       D2 cnpj wrong form  Banco I

## Summary

The envelope is clean, so the error-severity tests in `_staging_quality.yml` are a real guarantee. If they ever fire, our ingestion broke and stopping the build is correct.

The payload is not clean. D1, D2, and D3 all exist and are concentrated in specific institutions. D4 is empty once you look closely. The only apparent contradiction was the `0001-01-01` purchase-date placeholder, which is a D2 defect (a missing value in a date costume), not a disagreement between two real dates. The warn-severity tests in `_staging_quality.yml` count all of it and store the failing rows in `main_dbt_test__audit.*` on every build.

Where each piece lives:

1. **Detection, staging.** Envelope checks at severity error (ours). Payload checks at severity warn with stored failures (the provider's). This includes an explicit placeholder-date check, so the defect is named directly instead of surfacing by accident through a date-ordering rule.
2. **Resolution, intermediate.** Rows are repaired or flagged there, once, so no consumer has to redo it. Strip the CNPJ decimal tail, map IPC-A to IPCA, map `0001-01-01` to NULL, and flag what cannot be repaired in a `dq_flags` column (`missing:purchase_date`).
3. **Guarantee, canonical.** The output contract promises what survived. Structural fields are never null, enums are clean after mapping, and `dq_flags` tells the consumer what was admitted with a warning.